# JetMoE-8B Routing Trace Extraction (v2 — full rich parity with OLMoE, incl. MoA deep data)

Generates `jetmoe_routing_trace.json` (same 12 domain-balanced prompts as
`extract_routing_trace.ipynb`) plus `jetmoe_routing_trace_umap.json`.

**v2 upgrade — full parity.** This notebook now emits the **same rich deep-extraction schema as
`extract_routing_trace.ipynb`** (OLMoE) for the FFN side — top-level `layers` (RAW full-softmax
top-k, matching OLMoE), `router_matrices`, `hidden_vectors`, `expert_weights`, `expert_outputs`,
`embed_strip`, per-layer residual/RMSNorm flow — **plus** the deep data for JetMoE's defining
feature: **attention is itself Mixture-of-Experts (MoA)**.

**MoA (see `docs/research/model-architecture-jetmoe-deepseek-research.md` §2):** each of 8 attention
experts owns its Q (`experts.input_linear`) and output (`experts.output_linear`) projection but
they **share** one K/V projection (`kv_proj`); top-2 experts are gate-combined per token (plus a
single shared `experts.bias` on the merged output).

**Grouped-query attention (32 query heads / 16 K/V heads).** Each attention expert's Q projection
is sized `kv_channels × num_key_value_heads` = 128 × 16, so **one expert produces 16 query heads**
of 128 dims, and `kv_proj` produces **16 shared K/V heads**. In `modeling_jetmoe.py` the top-2
experts' query heads are stacked on the head axis (`query → [b, top_k×16 = 32, seq, 128]`) while
the shared K/V is repeated `top_k` times (`key_states.repeat(1, top_k, 1, 1)`), which is why
`JetMoeConfig.__post_init__` sets `num_attention_heads = num_key_value_heads × num_experts_per_tok
= 32`. Net: **32 query heads read 16 K/V heads, 2 query heads per K/V head — GQA (2:1)**, with the
grouping happening *across* the two selected experts (inside a single expert the mapping is 1:1).
The emitted `layer_flow` states this explicitly: `num_attention_heads` = 32 (model-wide query
heads), `num_query_heads_per_expert` = 16 (what `q_by_head` / `head_output_by_head` /
`attn_probs_all_heads` are indexed by), `num_kv_heads` = 16 (what `k_by_head` / `v_by_head` are
indexed by), `num_key_value_groups` = 2, `is_gqa = true`.

Captured per active (layer, attention-expert): the expert's Q/O weight downsamples
(`layer_flow.attn_expert_weights`) and its 16 per-head post-RoPE Q, attention maps (Q·Kᵀ→softmax),
and head outputs against the shared K/V (`layer_flow.attn_expert_flow`), alongside the shared
16 per-head K/V in each `per_layer` entry. `attention_routing` (8 experts, top-2, RAW weights)
drives the MoA heatmap/fan. `layer_flow.is_moa = true`, `has_qk_norm = false`.

The FFN experts are SwiGLU with a fused gate+up (`mlp.input_linear`, chunked) and `output_linear`
= down, plus a single shared `mlp.bias` on the merged output. A `reconcile=True` smoke pass
self-validates both the MoA attention recompute (vs the hooked `self_attention` output, including
`experts.bias`) and the FFN recompute (vs the real MoE delta) — printing RAW vs NORM rel_err so the
model's true combine convention is confirmed empirically before the full sweep. The reconcile is
also the empirical check on the head split: any other query/K-V head mapping fails it.

Run on a Colab A100 GPU runtime (8B params, ~16GB in bf16).


In [1]:
import importlib.util
import subprocess
import sys


def pip_install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])


if importlib.util.find_spec("torch") is None:
    pip_install("torch")

pip_install("transformers", "accelerate", "umap-learn", "numpy", "scikit-learn")

# JetMoE landed in mainline transformers very recently (JetMoeConfig / JetMoeForCausalLM,
# using Python 3.10+ union-type syntax and huggingface_hub's @strict config validation) --
# if the pip-published version is too old to have it, fall back to installing from source.
try:
    from transformers import JetMoeConfig  # noqa: F401
except ImportError:
    print("JetMoeConfig not found in installed transformers -- installing from GitHub main.")
    pip_install("git+https://github.com/huggingface/transformers.git")

print("Dependency installation complete.")

Dependency installation complete.


In [2]:
import json
import os
from collections import defaultdict

import numpy as np
import torch
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer, JetMoeConfig

MODEL_ID = "jetmoe/jetmoe-8b"
OUT_PATH = "jetmoe_routing_trace.json"
UMAP_OUT_PATH = "jetmoe_routing_trace_umap.json"
TOP_K_NEXT_TOKEN = 50

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# The HF repo's config.json predates JetMoE's mainline transformers integration and still
# uses the model's original field names (num_layers, moe_num_experts, moe_top_k) instead of
# the current JetMoeConfig field names (num_hidden_layers, num_local_experts,
# num_experts_per_tok) -- see docs/research/model-architecture-jetmoe-deepseek-research.md §2.1's
# "Important config.json caveat". Verify against the paper's known-correct numbers and patch
# if the loader didn't pick them up, rather than silently building the wrong-sized model.
# NOTE: num_attention_heads is deliberately NOT in EXPECTED -- this dict is splatted into
# JetMoeConfig(...) below and __post_init__ derives num_attention_heads itself; it gets its own
# check right after the load.
EXPECTED = {
    "num_hidden_layers": 24,
    "num_local_experts": 8,
    "num_experts_per_tok": 2,
    "hidden_size": 2048,
    "num_key_value_heads": 16,
    "kv_channels": 128,
}

try:
    config = AutoConfig.from_pretrained(MODEL_ID)
except Exception as e:  # noqa: BLE001 -- deliberately broad: @strict's exact error type for
    # legacy/unrecognized config keys isn't confirmed, so catch anything and fall back rather
    # than let one specific exception class slip through uncaught.
    print(f"AutoConfig.from_pretrained raised {e!r} (likely the legacy-key mismatch above). "
          f"Falling back to an explicitly-constructed JetMoeConfig with known-correct values.")
    config = JetMoeConfig(**EXPECTED, vocab_size=32000, max_position_embeddings=4096, intermediate_size=5632)

mismatches = {k: (getattr(config, k, "<missing>"), v) for k, v in EXPECTED.items() if getattr(config, k, None) != v}
if mismatches:
    print(f"Config field mismatch after load: {mismatches}")
    print("Patching to known-correct values from the paper/config.json before loading weights.")
    for k, v in EXPECTED.items():
        setattr(config, k, v)

for k, v in EXPECTED.items():
    assert getattr(config, k) == v, f"config.{k} = {getattr(config, k)}, expected {v} -- investigate before proceeding"

# ---- GQA check: 32 query heads over 16 shared K/V heads ----
# JetMoeConfig.__post_init__ derives num_attention_heads = num_key_value_heads * num_experts_per_tok
# = 16 * 2 = 32. That is not "16 heads": each of the top-2 selected attention experts contributes
# its own num_key_value_heads (16) query heads, and BOTH experts read the SAME 16 K/V heads --
# modeling_jetmoe.py stacks the experts' query heads on the head axis and does
# `key_states.repeat(1, top_k, 1, 1)` on the shared K/V. So the model runs 32 query heads against
# 16 K/V heads: grouped-query attention, 2 query heads per K/V head (paper Table 1 / research doc §2.2).
expected_query_heads = config.num_key_value_heads * config.num_experts_per_tok
if getattr(config, "num_attention_heads", None) != expected_query_heads:
    print(f"config.num_attention_heads = {getattr(config, 'num_attention_heads', '<missing>')}, "
          f"patching to {expected_query_heads} (num_key_value_heads x num_experts_per_tok, per JetMoeConfig.__post_init__).")
    config.num_attention_heads = expected_query_heads
assert config.num_attention_heads == 32 and config.num_key_value_heads == 16, (
    f"expected GQA 32 query heads / 16 K/V heads, got {config.num_attention_heads}/{config.num_key_value_heads}"
)

print("JetMoe config verified:", {k: getattr(config, k) for k in EXPECTED})
print(f"Attention is GQA: {config.num_attention_heads} query heads / {config.num_key_value_heads} K/V heads "
      f"({config.num_attention_heads // config.num_key_value_heads} query heads per K/V head), "
      f"head_dim {config.kv_channels}")

model, loading_info = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    config=config,
    dtype=torch.bfloat16,
    device_map="auto",
    output_loading_info=True,
)
model.eval()

assert not loading_info["missing_keys"], (
    f"Some model weights were NOT loaded from the checkpoint (randomly initialized "
    f"instead): {loading_info['missing_keys']}"
)
print(f"unexpected_keys (informational): {loading_info.get('unexpected_keys', [])}")

num_layers = config.num_hidden_layers
hidden_size = config.hidden_size
ffn_num_experts = config.num_local_experts        # JetMoeMoE.__init__ reads config.num_local_experts
ffn_top_k = config.num_experts_per_tok             # JetMoeMoE.__init__ reads config.num_experts_per_tok
attn_num_experts = config.num_local_experts        # JetMoeMoA.__init__ reads the SAME two config
attn_top_k = config.num_experts_per_tok            # fields -- attn/FFN expert counts coincide by construction, not coincidence

print(f"Loaded {MODEL_ID}: {num_layers} layers, FFN top-{ffn_top_k} of {ffn_num_experts}, "
      f"attention top-{attn_top_k} of {attn_num_experts}")

config.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/968 [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  493kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/266 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

unexpected_keys (informational): set()
Loaded jetmoe/jetmoe-8b: 24 layers, FFN top-2 of 8, attention top-2 of 8


In [3]:
# 6 domains x 2 short, trivia/completion-style prompts each = 12 total -- identical set to
# extract_routing_trace.ipynb's PROMPTS, so results are directly comparable across models.
PROMPTS = {
    "code": [
        "The most popular programming language for data science is",
        "A loop that never terminates is called an infinite",
    ],
    "math": [
        "The square root of sixteen is",
        "Two plus two equals",
    ],
    "biomedical": [
        "The organ that pumps blood throughout the human body is the",
        "White blood cells are a key part of the body's immune",
    ],
    "legal": [
        "The document that establishes the fundamental laws of the United States is called the",
        "A person accused of a crime is presumed",
    ],
    "creative_writing": [
        "It was a dark and stormy",
        "Roses are red, violets are",
    ],
    "conversational": [
        "Thank you so much, I really appreciate",
        "It was great catching up, see you",
    ],
}

domains = list(PROMPTS.keys())
total_prompts = sum(len(v) for v in PROMPTS.values())
print(f"Domains: {domains}")
print(f"Total prompts: {total_prompts}")
assert total_prompts == 12, f"Expected 12 prompts (6 x 2), got {total_prompts}"
for domain, examples in PROMPTS.items():
    assert len(examples) == 2, f"{domain} has {len(examples)} prompts, expected 2"

Domains: ['code', 'math', 'biomedical', 'legal', 'creative_writing', 'conversational']
Total prompts: 12


In [4]:
# ---- extraction helpers (downsampling) mirrored verbatim from extract_routing_trace.ipynb so
# the frontend math modals render JetMoE grids at the same resolution as OLMoE ----
def to_float(t):
    return t.detach().float().cpu()


def downsample_1d(vec, buckets):
    n = vec.shape[0]
    idx = torch.linspace(0, n, buckets + 1).round().long()
    return [round(vec[idx[i]: max(idx[i] + 1, idx[i + 1])].mean().item(), 5) for i in range(buckets)]


def downsample_2d(mat, rows, cols):
    R, C = mat.shape
    ridx = torch.linspace(0, R, rows + 1).round().long()
    cidx = torch.linspace(0, C, cols + 1).round().long()
    out = []
    for i in range(rows):
        r0, r1 = ridx[i].item(), max(ridx[i].item() + 1, ridx[i + 1].item())
        row_vals = []
        for j in range(cols):
            c0, c1 = cidx[j].item(), max(cidx[j].item() + 1, cidx[j + 1].item())
            row_vals.append(round(mat[r0:r1, c0:c1].mean().item(), 5))
        out.append(row_vals)
    return out


# downsample resolutions -- identical to extract_routing_trace.ipynb
ROUTER_GRID = (10, 12)
# MoA router downsample: FULL expert resolution on the row axis (8 rows = the 8 attention experts,
# one row each), same 12 column buckets as the FFN router. Row e must stay expert e -- the frontend
# draws this grid beside the 8-cell attention-router probability strip in the Attention modal.
ATTN_ROUTER_GRID = (attn_num_experts, ROUTER_GRID[1])
HIDDEN_STRIP = 20
EXPERT_GRID = (5, 5)
ATTN_GRID = (10, 10)
HEAD_STRIP = 6

# ---- MoA attention dims: grouped-query attention, 32 query heads / 16 K/V heads ----
# JetMoE attention is Mixture-of-Attention: each of the `attn_num_experts` experts owns its Q
# (experts.input_linear) and output (experts.output_linear) projection but they SHARE one K/V
# projection (kv_proj). Each expert's Q projection is sized kv_channels * num_key_value_heads,
# i.e. it produces `num_query_heads_per_expert` (= 16) query heads of `head_dim` (= 128) dims, and
# kv_proj produces `num_kv_heads` (= 16) K heads + 16 V heads.
#
# modeling_jetmoe.py stacks the top-`attn_top_k` experts' query heads on the head axis
# (query -> [b, top_k * 16 = 32, seq, 128]) and repeats the shared K/V top_k times
# (`key_states.repeat(1, self.top_k, 1, 1)`), so per token the model runs
# `num_query_heads` = 32 query heads against `num_kv_heads` = 16 K/V heads:
# `num_key_value_groups` = 2 query heads per K/V head -> GQA (2:1), exactly the 32/16 split in the
# paper's Table 1. The grouping is *across* the two selected experts; inside one expert the
# query-head -> K/V-head mapping is 1:1. All per-expert tensors below are therefore indexed
# 0..num_query_heads_per_expert-1, and the shared K/V tensors 0..num_kv_heads-1.
import torch.nn.functional as F  # noqa: E402 -- JetMoE's model-load cell doesn't import this

num_kv_heads = config.num_key_value_heads                      # 16 shared K/V heads (k_by_head / v_by_head)
num_query_heads = config.num_attention_heads                   # 32 query heads per token, model-wide
num_query_heads_per_expert = num_query_heads // attn_top_k     # 16 query heads owned by each attention expert
num_key_value_groups = num_query_heads // num_kv_heads         # 2 query heads share each K/V head
head_dim = getattr(config, "kv_channels", None) or (hidden_size // num_kv_heads)  # 128
try:
    rope_theta = config.rope_parameters["rope_theta"]
except Exception:
    rope_theta = getattr(config, "rope_theta", 10000.0)

assert num_query_heads == attn_top_k * num_query_heads_per_expert, "query heads must split evenly over the selected experts"
assert num_query_heads % num_kv_heads == 0 and num_key_value_groups > 1, (
    f"expected GQA (more query heads than K/V heads), got {num_query_heads} query / {num_kv_heads} K/V heads"
)
print(f"MoA attention: {attn_num_experts} experts top-{attn_top_k}; GQA {num_query_heads} query heads / "
      f"{num_kv_heads} K/V heads x {head_dim} dim ({num_key_value_groups} query heads per K/V head), "
      f"{num_query_heads_per_expert} query heads per attention expert; rope_theta={rope_theta}")

# Fail fast if any model-internal attribute path this notebook depends on differs from the
# modeling source (JetMoeTopKGating.layer, JetMoeMoA.{input_linear,output_linear,router,kv_proj,bias}).
# A wrong path here would otherwise surface as a confusing error deep inside the sweep -- catch it
# now, on layer 0, and print the real attribute names so they can be corrected quickly.
_l0 = model.model.layers[0]
assert hasattr(_l0.mlp, "router") and hasattr(_l0.mlp.router, "layer") and hasattr(_l0.mlp.router.layer, "weight"), \
    f"expected mlp.router.layer.weight; mlp attrs={list(dict(_l0.mlp.named_children()))}"
assert hasattr(_l0.mlp, "input_linear") and hasattr(_l0.mlp, "output_linear") and hasattr(_l0.mlp, "bias"), \
    f"expected mlp.input_linear/output_linear/bias; mlp attrs={list(dict(_l0.mlp.named_children()))}"
assert hasattr(_l0.self_attention, "kv_proj") and hasattr(_l0.self_attention, "experts"), \
    f"expected self_attention.kv_proj + .experts; attrs={list(dict(_l0.self_attention.named_children()))}"
_ex = _l0.self_attention.experts
assert hasattr(_ex, "input_linear") and hasattr(_ex, "output_linear") and hasattr(_ex, "router"), \
    f"expected experts.input_linear/output_linear/router; attrs={list(dict(_ex.named_children()))}"
# JetMoeMoA.reduce ends with `layer_output = layer_output + self.bias`, so the hooked self_attention
# output includes this shared bias -- the MoA reconciliation below has to add it too.
assert hasattr(_ex, "bias"), f"expected experts.bias (shared MoA output bias); attrs={list(dict(_ex.named_parameters()))}"

# ---- structural proof of the 32/16 GQA split (weight shapes, not just config numbers) ----
_q_out, _q_in = tuple(_ex.input_linear.weight[0].shape)
_o_out, _o_in = tuple(_ex.output_linear.weight[0].shape)
_kv_out, _kv_in = tuple(_l0.self_attention.kv_proj.weight.shape)
assert (_q_out, _q_in) == (num_query_heads_per_expert * head_dim, hidden_size), \
    f"attn expert W_q is {(_q_out, _q_in)}, expected {(num_query_heads_per_expert * head_dim, hidden_size)} " \
    f"({num_query_heads_per_expert} query heads x {head_dim})"
assert (_o_out, _o_in) == (hidden_size, num_query_heads_per_expert * head_dim), \
    f"attn expert W_o is {(_o_out, _o_in)}, expected {(hidden_size, num_query_heads_per_expert * head_dim)}"
assert (_kv_out, _kv_in) == (2 * num_kv_heads * head_dim, hidden_size), \
    f"shared kv_proj is {(_kv_out, _kv_in)}, expected {(2 * num_kv_heads * head_dim, hidden_size)} " \
    f"(K and V, {num_kv_heads} heads x {head_dim} each)"

print("verified module paths: mlp.router.layer.weight, mlp.input_linear/output_linear/bias, "
      "self_attention.kv_proj, self_attention.experts.{input_linear,output_linear,router,bias}")
print(f"ffn input_linear.weight[0] shape={tuple(_l0.mlp.input_linear.weight[0].shape)} (expect [2*ffn, hidden]); "
      f"MoA input_linear.weight[0]={(_q_out, _q_in)} = {num_query_heads_per_expert} query heads x {head_dim}; "
      f"kv_proj.weight={(_kv_out, _kv_in)} = K+V, {num_kv_heads} heads x {head_dim} -> "
      f"{attn_top_k} experts x {num_query_heads_per_expert} = {num_query_heads} query heads over {num_kv_heads} K/V heads")


MoA attention: 8 experts top-2; GQA 32 query heads / 16 K/V heads x 128 dim (2 query heads per K/V head), 16 query heads per attention expert; rope_theta=10000.0
verified module paths: mlp.router.layer.weight, mlp.input_linear/output_linear/bias, self_attention.kv_proj, self_attention.experts.{input_linear,output_linear,router,bias}
ffn input_linear.weight[0] shape=(11264, 2048) (expect [2*ffn, hidden]); MoA input_linear.weight[0]=(2048, 2048) = 16 query heads x 128; kv_proj.weight=(4096, 2048) = K+V, 16 heads x 128 -> 2 experts x 16 = 32 query heads over 16 K/V heads


## MoA attention-router weights (standalone, no forward pass)

Emits **only** `attention_routing.router_matrices` -- the Mixture-of-Attention router's own weight
matrix, downsampled per layer. The frontend's Attention modal step 2 draws it as a grid; without it
the multiply there renders as a bare `· W_router →` arrow.

**Run this cell after cells 1, 2 and 4 only** (deps, model load, helpers/dims). It needs no tokenizer,
no prompts and no forward pass -- a router weight is a model parameter, identical for every prompt --
so it is the cheap way to add this field to trace files that were extracted before the field existed.
Output is ~30 KB, versus the ~100 MB the full 12-prompt sweep produces.

Merge it into the already-published per-prompt files locally with:

```
node scripts/merge-attn-router.cjs <downloaded>.json public/data/JETMoe/jetmoe_routing_trace
```

The full sweep (cells 5-7 below) emits the identical numbers inline, so re-running everything works
too -- this cell just avoids paying for it.

In [5]:
# ---- STANDALONE: MoA attention-router weight matrices only ----
# Needs cells 1, 2 and 4 (deps, model, helpers/dims). No tokenizer, no prompts, no forward pass.
ATTN_ROUTER_OUT = "jetmoe_attn_router_matrices.json"

_w0 = model.model.layers[0].self_attention.experts.router.layer.weight
assert tuple(_w0.shape) == (attn_num_experts, hidden_size), (
    f"MoA router weight is {tuple(_w0.shape)}, expected {(attn_num_experts, hidden_size)} -- "
    f"check self_attention.experts.router.layer is still JetMoeTopKGating's nn.Linear"
)

attn_router_matrices = [
    downsample_2d(
        to_float(model.model.layers[li].self_attention.experts.router.layer.weight),
        *ATTN_ROUTER_GRID,
    )
    for li in range(num_layers)
]

# Row e must stay expert e (rows unbucketed), or the grid no longer aligns with the 8-cell
# probability strip the frontend draws beside it.
assert len(attn_router_matrices) == num_layers
assert all(len(m) == attn_num_experts and len(m[0]) == ATTN_ROUTER_GRID[1] for m in attn_router_matrices)
# Distinctness guard: the FFN router is a different matrix in scope with a compatible shape, and
# quietly shipping THAT one here is the single wrong outcome that would still render plausibly.
_ffn0 = downsample_2d(to_float(model.model.layers[0].mlp.router.layer.weight), *ATTN_ROUTER_GRID)
assert attn_router_matrices[0] != _ffn0, "extracted the FFN router, not the MoA router"

with open(ATTN_ROUTER_OUT, "w", encoding="utf-8") as f:
    json.dump({
        "model_id": MODEL_ID,
        "num_layers": num_layers,
        "attn_num_experts": attn_num_experts,
        "hidden_size": hidden_size,
        "grid": list(ATTN_ROUTER_GRID),
        "router_matrices": attn_router_matrices,
    }, f)

print(f"wrote {ATTN_ROUTER_OUT} ({os.path.getsize(ATTN_ROUTER_OUT) / 1e3:.1f} KB): {num_layers} layers x "
      f"{attn_num_experts} experts x {ATTN_ROUTER_GRID[1]} column buckets "
      f"(true weight shape {tuple(_w0.shape)} per layer)")
print("layer 1 / expert 1 row:", attn_router_matrices[0][0])

wrote jetmoe_attn_router_matrices.json (21.8 KB): 24 layers x 8 experts x 12 column buckets (true weight shape (8, 2048) per layer)
layer 1 / expert 1 row: [0.01106, -0.00265, 0.0121, 0.00889, 0.0189, 0.00793, 0.00161, -0.00376, 0.00766, 0.00739, -0.01276, 0.00674]


In [6]:
def _rotate_half(x):
    x1, x2 = x[..., : x.shape[-1] // 2], x[..., x.shape[-1] // 2:]
    return torch.cat((-x2, x1), dim=-1)


def _rope_cos_sin(seq_len, hd, base):
    inv_freq = 1.0 / (base ** (torch.arange(0, hd, 2, dtype=torch.float32) / hd))
    freqs = torch.outer(torch.arange(seq_len, dtype=torch.float32), inv_freq)
    emb = torch.cat((freqs, freqs), dim=-1)
    return emb.cos(), emb.sin()


def _moa_expert_attention(q_full, k_full, v_full, cos, sin, causal_mask, nq, nkv, hd):
    """One attention expert's per-head Q.Kᵀ→softmax→×V using its own Q and the SHARED K/V.

    q_full: [seq, nq*hd] (this expert's `nq` query heads). k/v_full: [seq, nkv*hd] (the model's
    shared K/V heads). Grouped-query attention: query head h reads K/V head h // (nq // nkv) --
    with JetMoE's numbers nq == nkv == 16 inside one expert, so the mapping here is 1:1; the real
    2:1 grouping is that BOTH selected experts' 16 query heads (32 in total per token) read these
    same 16 shared K/V heads, which is what `key_states.repeat(1, top_k, 1, 1)` does in
    modeling_jetmoe.py. Returns (attn_maps, head_outputs, q_rope_by_head, q_prerope_by_head), each
    a list of nq tensors. K gets RoPE (recomputed here; identical to the shared k_by_head).
    """
    assert nq % nkv == 0, f"query heads {nq} must be a multiple of K/V heads {nkv}"
    group = nq // nkv  # query heads per K/V head, within this expert
    scale = 1.0 / (hd ** 0.5)
    maps, head_outs, q_rope_bh, q_pre_bh = [], [], [], []
    for h in range(nq):
        kvh = h // group  # the shared K/V head this query head is grouped onto
        qh = q_full[:, h * hd:(h + 1) * hd]
        kh = k_full[:, kvh * hd:(kvh + 1) * hd]
        vh = v_full[:, kvh * hd:(kvh + 1) * hd]
        qh_r = qh * cos + _rotate_half(qh) * sin
        kh_r = kh * cos + _rotate_half(kh) * sin
        scores = (qh_r @ kh_r.transpose(-1, -2)) * scale + causal_mask
        amap = torch.softmax(scores, dim=-1)
        maps.append(amap)
        head_outs.append(amap @ vh)
        q_rope_bh.append(qh_r)
        q_pre_bh.append(qh)
    return maps, head_outs, q_rope_bh, q_pre_bh


def _build_routing_trace(logits, top_k, num_experts, seq_len):
    """Raw full-softmax top-k (NOT renormalized) -- same convention as OLMoE/DeepSeek."""
    assert logits.shape == (seq_len, num_experts), f"logits {tuple(logits.shape)} != ({seq_len},{num_experts})"
    probs = torch.softmax(logits, dim=-1)
    topk = torch.topk(probs, k=top_k, dim=-1)
    tokens_trace = []
    for t in range(seq_len):
        tokens_trace.append({
            "token_index": t,
            "top_experts": topk.indices[t].tolist(),
            "top_weights": [round(w, 5) for w in topk.values[t].tolist()],
            "all_probs": [round(p, 5) for p in probs[t].tolist()],
        })
    return tokens_trace


def extract_for_prompt(prompt, reconcile=False):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    token_ids = inputs["input_ids"][0].tolist()
    token_strs = [tokenizer.decode([tid]) for tid in token_ids]
    seq_len = len(token_ids)

    # hooks: the two RMSNorms (ln1 = attention/router input, ln2 = FFN/router input) and the whole
    # attention module output (the gate-combined MoA result + shared MoA bias, for reconciliation).
    ln1_outputs, ln2_outputs, attn_outputs = {}, {}, {}
    hooks = []

    def make_post_hook(store, li):
        def hook(module, args, output):
            val = output[0] if isinstance(output, tuple) else output
            store[li] = val[0].detach().float().cpu()  # drop batch dim -> [seq, *]
        return hook

    for li, layer in enumerate(model.model.layers):
        hooks.append(layer.input_layernorm.register_forward_hook(make_post_hook(ln1_outputs, li)))
        hooks.append(layer.post_attention_layernorm.register_forward_hook(make_post_hook(ln2_outputs, li)))
        hooks.append(layer.self_attention.register_forward_hook(make_post_hook(attn_outputs, li)))

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)

    for h in hooks:
        h.remove()

    silu = torch.nn.functional.silu
    causal_mask = torch.triu(torch.full((seq_len, seq_len), float("-inf")), diagonal=1)
    rope_cos, rope_sin = _rope_cos_sin(seq_len, head_dim, rope_theta)

    # ---- FFN + attention routing (recomputed from the hooked RMSNorm outputs + router weights;
    # ln1_out feeds the MoA router, ln2_out feeds the FFN router) ----
    ffn_layers_trace, attn_layers_trace = [], []
    for li in range(num_layers):
        layer = model.model.layers[li]
        ffn_logits = F.linear(ln2_outputs[li], to_float(layer.mlp.router.layer.weight))
        attn_logits = F.linear(ln1_outputs[li], to_float(layer.self_attention.experts.router.layer.weight))
        ffn_layers_trace.append({"layer": li, "tokens": _build_routing_trace(ffn_logits, ffn_top_k, ffn_num_experts, seq_len)})
        attn_layers_trace.append({"layer": li, "tokens": _build_routing_trace(attn_logits, attn_top_k, attn_num_experts, seq_len)})

    # ---- next-token candidates ----
    next_token_logits = outputs.logits[0, -1, :]
    next_token_probs = torch.softmax(next_token_logits.float(), dim=-1)
    top_next = torch.topk(next_token_probs, k=TOP_K_NEXT_TOKEN)
    next_token_candidates = [
        {"token": tokenizer.decode([tid]), "prob": round(p, 6)}
        for p, tid in zip(top_next.values.tolist(), top_next.indices.tolist())
    ]

    # ---- router weight downsamples (FFN + MoA) + hidden vectors (FFN router input = ln2_out) ----
    # Both routers are model parameters, so these are prompt-independent -- emitted per prompt anyway
    # so each split prompt file stays self-contained (usePromptFlow fetches exactly one of them).
    router_matrices, attn_router_matrices, hidden_vectors = [], [], []
    for li in range(num_layers):
        _layer = model.model.layers[li]
        router_matrices.append(downsample_2d(to_float(_layer.mlp.router.layer.weight), *ROUTER_GRID))
        # MoA router weight [attn_num_experts, hidden] -- rows deliberately unbucketed, see ATTN_ROUTER_GRID.
        attn_router_matrices.append(downsample_2d(to_float(_layer.self_attention.experts.router.layer.weight), *ATTN_ROUTER_GRID))
        hidden_vectors.append([downsample_1d(ln2_outputs[li][t], HIDDEN_STRIP) for t in range(seq_len)])

    # ---- FFN expert weights + recomputed outputs (JetMoeMoE: input_linear fuses gate+up as the
    # OUTPUT halves -> chunk(2, dim=0) on each expert's [2*ffn, hidden] weight; output_linear=down;
    # a single shared self.bias is added to the merged output, so it's NOT part of a per-expert out) ----
    def ffn_expert_weight(li, e):
        mlp = model.model.layers[li].mlp
        fused = to_float(mlp.input_linear.weight[e])       # [2*ffn, hidden]
        ffn = fused.shape[0] // 2
        gate_w, up_w = fused[:ffn, :], fused[ffn:, :]
        down_w = to_float(mlp.output_linear.weight[e])     # [hidden, ffn]
        return gate_w, up_w, down_w

    active_ffn = sorted({(lt["layer"], e) for lt in ffn_layers_trace for tt in lt["tokens"] for e in tt["top_experts"]})
    expert_weights, ffn_wcache = {}, {}
    for li, e in active_ffn:
        gate_w, up_w, down_w = ffn_expert_weight(li, e)
        ffn_wcache[(li, e)] = (gate_w, up_w, down_w)
        expert_weights[f"{li}_{e}"] = {
            "gate": downsample_2d(gate_w, *EXPERT_GRID),
            "up": downsample_2d(up_w, *EXPERT_GRID),
            "down": downsample_2d(down_w, *EXPERT_GRID),
        }
    intermediate_size = ffn_wcache[active_ffn[0]][0].shape[0] if active_ffn else None

    expert_outputs = {}
    for lt in ffn_layers_trace:
        li = lt["layer"]
        for t, tt in enumerate(lt["tokens"]):
            h_t = ln2_outputs[li][t]
            for e in tt["top_experts"]:
                gate_w, up_w, down_w = ffn_wcache[(li, e)]
                out = F.linear(silu(F.linear(h_t, gate_w)) * F.linear(h_t, up_w), down_w)
                expert_outputs[f"{t}_{li}_{e}"] = downsample_1d(out, HIDDEN_STRIP)

    # ---- embeddings ----
    hidden_states_all = [h[0].detach().float().cpu() for h in outputs.hidden_states]
    embed_vec = to_float(model.model.embed_tokens.weight)[torch.tensor(token_ids)]
    embed_strip = [downsample_1d(embed_vec[t], HIDDEN_STRIP) for t in range(seq_len)]

    # ---- per-layer flow + MoA attention-expert deep data ----
    active_attn = sorted({(lt["layer"], e) for lt in attn_layers_trace for tt in lt["tokens"] for e in tt["top_experts"]})
    active_attn_by_layer = {li: [e for (l2, e) in active_attn if l2 == li] for li in range(num_layers)}

    per_layer_flow = []
    attn_expert_weights, attn_expert_flow = {}, {}
    recon_errs = []  # best-convention rel_err per reconciled quantity; the smoke asserts on this
    for li in range(num_layers):
        layer = model.model.layers[li]
        attn = layer.self_attention
        ln1 = ln1_outputs[li]

        # shared K/V (one projection for all experts): kv_proj -> chunk into K, V, each [seq, nkv*hd]
        kv = F.linear(ln1, to_float(attn.kv_proj.weight))     # [seq, 2*num_kv_heads*head_dim]
        k_full, v_full = kv.chunk(2, dim=-1)                   # [seq, num_kv_heads*head_dim] each
        # The attention modal prints these widths as the "K raw" / "V raw" dim labels, computed
        # from num_kv_heads x head_dim -- which equals hidden_size on JetMoE. That is a config
        # coincidence, not an identity, and a wrong width renders perfectly plausibly, so assert it.
        assert k_full.shape[-1] == num_kv_heads * head_dim == hidden_size, (
            f"K raw width {k_full.shape[-1]} != {num_kv_heads}x{head_dim} or != hidden {hidden_size}")
        kv_w = to_float(attn.kv_proj.weight)
        kv_split = kv_w.shape[0] // 2

        # the 16 shared K/V heads, reused by every attention expert's query heads (GQA)
        k_by_head, v_by_head, k_by_head_prerope = [], [], []
        for h in range(num_kv_heads):
            kh = k_full[:, h * head_dim:(h + 1) * head_dim]
            vh = v_full[:, h * head_dim:(h + 1) * head_dim]
            kh_r = kh * rope_cos + _rotate_half(kh) * rope_sin
            k_by_head_prerope.append([downsample_1d(kh[t], HEAD_STRIP) for t in range(seq_len)])
            k_by_head.append([downsample_1d(kh_r[t], HEAD_STRIP) for t in range(seq_len)])
            v_by_head.append([downsample_1d(vh[t], HEAD_STRIP) for t in range(seq_len)])

        # per active attention expert: its `num_query_heads_per_expert` query heads, their attention
        # maps against the shared K/V, head outputs, and its own O projection. Query head h of this
        # expert reads shared K/V head h // (num_query_heads_per_expert // num_kv_heads) -- 1:1 with
        # JetMoE's numbers, so the per-head arrays below line up index-for-index with k_by_head /
        # v_by_head and no per-entry mapping table is stored (see layer_flow's GQA scalars).
        expert_attn_out = {}  # e -> [seq, hidden] full-precision (for reconciliation)
        for e in active_attn_by_layer[li]:
            q_full = F.linear(ln1, to_float(attn.experts.input_linear.weight[e]))  # [seq, nq*hd]
            assert q_full.shape[-1] == num_query_heads_per_expert * head_dim, (
                f"Q raw width {q_full.shape[-1]} != {num_query_heads_per_expert}x{head_dim}")
            maps, head_outs, q_rope_bh, q_pre_bh = _moa_expert_attention(
                q_full, k_full, v_full, rope_cos, rope_sin, causal_mask,
                num_query_heads_per_expert, num_kv_heads, head_dim)
            concat = torch.cat(head_outs, dim=-1)                                  # [seq, nq*hd]
            o_w = to_float(attn.experts.output_linear.weight[e])                   # [hidden, nq*hd]
            expert_attn_out[e] = F.linear(concat, o_w)
            attn_expert_weights[f"{li}_{e}"] = {
                "q": downsample_2d(to_float(attn.experts.input_linear.weight[e]), *ATTN_GRID),
                "o": downsample_2d(o_w, *ATTN_GRID),
            }
            attn_expert_flow[f"{li}_{e}"] = {
                # pre-split Q at its full nq*head_dim width -- the "Q raw" block in the attention
                # math modal's step 1. Per expert, because W_q is (K/V raw live on per_layer).
                "q_raw": [downsample_1d(q_full[t], HIDDEN_STRIP) for t in range(seq_len)],
                "attn_probs_all_heads": [[[round(v, 5) for v in row] for row in maps[h].tolist()] for h in range(num_query_heads_per_expert)],
                "q_by_head": [[downsample_1d(q_rope_bh[h][t], HEAD_STRIP) for t in range(seq_len)] for h in range(num_query_heads_per_expert)],
                "q_by_head_prerope": [[downsample_1d(q_pre_bh[h][t], HEAD_STRIP) for t in range(seq_len)] for h in range(num_query_heads_per_expert)],
                "head_output_by_head": [[downsample_1d(head_outs[h][t], HEAD_STRIP) for t in range(seq_len)] for h in range(num_query_heads_per_expert)],
            }

        layer_in = hidden_states_all[li]
        layer_out = hidden_states_all[li + 1]
        attn_out = attn_outputs[li]
        after_attn_residual = layer_in + attn_out
        moe_out = layer_out - after_attn_residual

        per_layer_flow.append({
            "ln1_weight": downsample_1d(to_float(layer.input_layernorm.weight), HIDDEN_STRIP),
            "ln1_out": [downsample_1d(ln1[t], HIDDEN_STRIP) for t in range(seq_len)],
            "k_weight": downsample_2d(kv_w[:kv_split, :], *ATTN_GRID),
            "v_weight": downsample_2d(kv_w[kv_split:, :], *ATTN_GRID),
            # pre-split K/V at their full num_kv_heads*head_dim width -- the "K raw" / "V raw"
            # blocks in the attention math modal's step 1. Shared by every attention expert.
            "k_raw": [downsample_1d(k_full[t], HIDDEN_STRIP) for t in range(seq_len)],
            "v_raw": [downsample_1d(v_full[t], HIDDEN_STRIP) for t in range(seq_len)],
            "k_by_head": k_by_head, "v_by_head": v_by_head, "k_by_head_prerope": k_by_head_prerope,
            "active_attn_experts": active_attn_by_layer[li],
            "attn_output": [downsample_1d(attn_out[t], HIDDEN_STRIP) for t in range(seq_len)],
            "after_attn_residual": [downsample_1d(after_attn_residual[t], HIDDEN_STRIP) for t in range(seq_len)],
            "ln2_weight": downsample_1d(to_float(layer.post_attention_layernorm.weight), HIDDEN_STRIP),
            "ln2_out": [downsample_1d(ln2_outputs[li][t], HIDDEN_STRIP) for t in range(seq_len)],
            "moe_output": [downsample_1d(moe_out[t], HIDDEN_STRIP) for t in range(seq_len)],
            "layer_output": [downsample_1d(layer_out[t], HIDDEN_STRIP) for t in range(seq_len)],
        })

        # ---- self-validation: MoA reconstruction (gate-combined top-k experts + shared MoA bias)
        # vs the hooked self_attention output. This is also the empirical check on the GQA head
        # split: a wrong query-head -> K/V-head mapping does not reproduce the real output. ----
        if reconcile and li == 0:
            moa_bias = to_float(attn.experts.bias)
            for t in [0, seq_len - 1]:
                tt = attn_layers_trace[li]["tokens"][t]
                routed = sum(w * expert_attn_out[e][t] for e, w in zip(tt["top_experts"], tt["top_weights"]))
                wsum = sum(tt["top_weights"])
                real = attn_out[t]
                rel_raw = (routed + moa_bias - real).norm().item() / (real.norm().item() + 1e-9)
                rel_norm = (routed / wsum + moa_bias - real).norm().item() / (real.norm().item() + 1e-9)
                recon_errs.append(min(rel_raw, rel_norm))
                print(f"    [reconcile-MoA] layer {li} tok {t}: RAW+bias rel_err={rel_raw:.4f} | NORM+bias rel_err={rel_norm:.4f} "
                      f"(whichever is ~0 is the model's real attention-combine convention)")

    # ---- self-validation: FFN reconstruction (top-k experts + shared bias) vs real MoE delta ----
    if reconcile:
        li = 0
        for t in [0, seq_len - 1]:
            tt = ffn_layers_trace[li]["tokens"][t]
            h_t = ln2_outputs[li][t]
            routed_raw = sum(w * F.linear(silu(F.linear(h_t, ffn_wcache[(li, e)][0])) * F.linear(h_t, ffn_wcache[(li, e)][1]), ffn_wcache[(li, e)][2])
                             for e, w in zip(tt["top_experts"], tt["top_weights"]))
            wsum = sum(tt["top_weights"])
            bias = to_float(model.model.layers[li].mlp.bias)
            real = hidden_states_all[li + 1][t] - (hidden_states_all[li][t] + attn_outputs[li][t])
            rel_raw = (routed_raw + bias - real).norm().item() / (real.norm().item() + 1e-9)
            rel_norm = (routed_raw / wsum + bias - real).norm().item() / (real.norm().item() + 1e-9)
            recon_errs.append(min(rel_raw, rel_norm))
            print(f"    [reconcile-FFN] layer {li} tok {t}: RAW+bias rel_err={rel_raw:.4f} | NORM+bias rel_err={rel_norm:.4f} "
                  f"(whichever is ~0 is the model's real FFN-combine convention)")

    trace = {
        "prompt": prompt,
        "model_id": MODEL_ID,
        "num_layers": num_layers,
        "num_experts": ffn_num_experts,
        "top_k_experts": ffn_top_k,
        "hidden_size": hidden_size,
        "intermediate_size": intermediate_size,
        "tokens": [{"index": i, "text": s} for i, s in enumerate(token_strs)],
        "layers": ffn_layers_trace,
        "attention_routing": {"num_experts": attn_num_experts, "top_k": attn_top_k, "layers": attn_layers_trace,
                              "router_matrices": attn_router_matrices, "grid": list(ATTN_ROUTER_GRID)},
        "next_token_candidates": next_token_candidates,
        "router_matrices": router_matrices,
        "hidden_vectors": hidden_vectors,
        "expert_weights": expert_weights,
        "expert_outputs": expert_outputs,
        "grid_dims": {"router": list(ROUTER_GRID), "hidden_strip": HIDDEN_STRIP, "expert": list(EXPERT_GRID)},
        "layer_flow": {
            # GQA: 32 query heads per token (attn_top_k selected experts x 16 query heads each) over
            # 16 shared K/V heads. `num_attention_heads` is the model-wide query-head count; the
            # per-expert tensors in attn_expert_flow are indexed by num_query_heads_per_expert and
            # the shared k_by_head/v_by_head by num_kv_heads.
            "num_attention_heads": num_query_heads,
            "num_query_heads": num_query_heads,
            "num_query_heads_per_expert": num_query_heads_per_expert,
            "num_kv_heads": num_kv_heads,
            "num_key_value_groups": num_key_value_groups,
            "is_gqa": True,
            "attn_num_experts": attn_num_experts,
            "attn_top_k": attn_top_k,
            "head_dim": head_dim,
            "has_qk_norm": False,
            "is_moa": True,
            "embed_strip": embed_strip,
            "per_layer": per_layer_flow,
            "attn_expert_weights": attn_expert_weights,
            "attn_expert_flow": attn_expert_flow,
            "grid_dims": {"attn": list(ATTN_GRID)},
        },
    }
    if reconcile:
        trace["_reconcile"] = recon_errs  # smoke-only; the sweep calls reconcile=False so it never ships
    return trace


In [7]:
# Can't execute this notebook locally to verify the hook wiring / MoA math -- run one prompt
# first and sanity-check shapes and the reconciliation diagnostics before the full 12-prompt
# sweep. reconcile=True self-validates the untestable deep extraction: only the correct combine
# convention AND the correct GQA query-head -> K/V-head mapping reproduce the real outputs, so a
# small rel_err confirms the MoA per-expert attention recompute (32 query heads over 16 shared
# K/V heads, + the shared experts.bias) AND the FFN expert recompute are wired correctly.
smoke_trace = extract_for_prompt(PROMPTS["math"][0], reconcile=True)
print(f"tokens: {[t['text'] for t in smoke_trace['tokens']]}")

ffn_layers = smoke_trace["layers"]
attn_layers = smoke_trace["attention_routing"]["layers"]
print(f"ffn layers: {len(ffn_layers)} / attn layers: {len(attn_layers)} (expect {num_layers} each)")
print(f"layer 0 tok 0 ffn top_experts: {ffn_layers[0]['tokens'][0]['top_experts']} "
      f"(sum top_weights {sum(ffn_layers[0]['tokens'][0]['top_weights']):.3f} raw; all_probs {sum(ffn_layers[0]['tokens'][0]['all_probs']):.3f})")
print(f"layer 0 tok 0 attn top_experts: {attn_layers[0]['tokens'][0]['top_experts']} "
      f"(sum top_weights {sum(attn_layers[0]['tokens'][0]['top_weights']):.3f} raw)")

lf = smoke_trace["layer_flow"]
pl0 = lf["per_layer"][0]
print(f"layer_flow: is_moa={lf['is_moa']}, is_gqa={lf['is_gqa']}, "
      f"{lf['num_attention_heads']} query heads / {lf['num_kv_heads']} K/V heads x {lf['head_dim']} dim "
      f"({lf['num_key_value_groups']} query heads per K/V head), "
      f"{lf['num_query_heads_per_expert']} query heads per attention expert, "
      f"attn {lf['attn_num_experts']} experts top-{lf['attn_top_k']}, has_qk_norm={lf['has_qk_norm']}")
print(f"per_layer[0] active_attn_experts: {pl0['active_attn_experts']}; "
      f"shared k_by_head heads: {len(pl0['k_by_head'])} (expect {num_kv_heads})")
print(f"attn_expert_weights pairs: {len(lf['attn_expert_weights'])}; attn_expert_flow pairs: {len(lf['attn_expert_flow'])}")
sample_key = next(iter(lf["attn_expert_flow"]))
fl = lf["attn_expert_flow"][sample_key]
print(f"attn_expert_flow['{sample_key}'] maps: {len(fl['attn_probs_all_heads'])} query heads x "
      f"{len(fl['attn_probs_all_heads'][0])}x{len(fl['attn_probs_all_heads'][0][0])} "
      f"(expect {num_query_heads_per_expert} x seq x seq)")
print(f"ffn expert_weights pairs: {len(smoke_trace['expert_weights'])}; expert_outputs: {len(smoke_trace['expert_outputs'])}; "
      f"intermediate_size={smoke_trace['intermediate_size']}")
print(f"top next-token prediction: {smoke_trace['next_token_candidates'][0]}")

# causal-structure eyeball on a real MoA attention map (sample expert, query head 0)
amap = fl["attn_probs_all_heads"][0]
upper = sum(amap[i][j] for i in range(len(amap)) for j in range(i + 1, len(amap)))
print(f"MoA sample expert query head 0 strictly-upper-tri attn mass: {upper:.4f} (expect ~0 for causal masking)")

assert len(ffn_layers) == num_layers and len(attn_layers) == num_layers
assert len(ffn_layers[0]["tokens"][0]["top_experts"]) == ffn_top_k
assert len(attn_layers[0]["tokens"][0]["top_experts"]) == attn_top_k
# GQA bookkeeping: shared K/V tensors are indexed by num_kv_heads (16), per-expert query tensors by
# num_query_heads_per_expert (16), and the two selected experts together make up the model's
# num_attention_heads (32) query heads over those same 16 K/V heads.
assert len(pl0["k_by_head"]) == num_kv_heads and len(pl0["v_by_head"]) == num_kv_heads
assert len(fl["attn_probs_all_heads"]) == num_query_heads_per_expert
assert len(fl["q_by_head"]) == num_query_heads_per_expert and len(fl["head_output_by_head"]) == num_query_heads_per_expert
assert lf["is_moa"] is True and lf["is_gqa"] is True
assert lf["num_attention_heads"] == 32 and lf["num_kv_heads"] == 16 and lf["num_key_value_groups"] == 2
assert lf["num_attention_heads"] == lf["attn_top_k"] * lf["num_query_heads_per_expert"]
assert upper < 0.05, "MoA attention map is not causal -- check the causal mask / RoPE / head reshape"
# HALT before the expensive 12-prompt sweep if MoA or FFN recompute doesn't reconstruct the model:
assert max(smoke_trace["_reconcile"]) < 0.1, (
    f"RECONCILE FAILED {smoke_trace['_reconcile']} -- MoA attention (32 query heads / 16 shared K/V heads "
    f"+ experts.bias) and/or FFN recompute does not match the real model output. Do NOT run the full "
    f"sweep; fix the extraction first (see notebook README/handoff)."
)

moa_errs, ffn_errs = smoke_trace["_reconcile"][:2], smoke_trace["_reconcile"][2:]
fmt = lambda errs: [f"{x:.2e}" for x in errs]  # noqa: E731 -- rel_errs are ~1e-3, round() would flatten them to 0.0
print(f"Smoke test passed. reconcile best-convention rel_errs -- MoA: {fmt(moa_errs)} | FFN: {fmt(ffn_errs)}")
print("CHECK BY EYE: the MoA rel_errs should be the same order of magnitude as the FFN ones (both should be "
      "tiny -- roughly 1e-3 or below in bf16). The 0.1 assert above is only a coarse HALT gate; an MoA error "
      "much larger than the FFN error still means the attention recompute is off (query-head -> K/V-head "
      "mapping, RoPE, or the shared experts.bias) even though the assert passed.")


    [reconcile-MoA] layer 0 tok 0: RAW+bias rel_err=0.5509 | NORM+bias rel_err=0.0025 (whichever is ~0 is the model's real attention-combine convention)
    [reconcile-MoA] layer 0 tok 6: RAW+bias rel_err=0.6327 | NORM+bias rel_err=0.0037 (whichever is ~0 is the model's real attention-combine convention)
    [reconcile-FFN] layer 0 tok 0: RAW+bias rel_err=0.3210 | NORM+bias rel_err=0.0030 (whichever is ~0 is the model's real FFN-combine convention)
    [reconcile-FFN] layer 0 tok 6: RAW+bias rel_err=0.1290 | NORM+bias rel_err=0.0041 (whichever is ~0 is the model's real FFN-combine convention)
tokens: ['<s>', 'The', 'square', 'root', 'of', 'sixteen', 'is']
ffn layers: 24 / attn layers: 24 (expect 24 each)
layer 0 tok 0 ffn top_experts: [1, 6] (sum top_weights 0.510 raw; all_probs 1.000)
layer 0 tok 0 attn top_experts: [2, 0] (sum top_weights 0.449 raw)
layer_flow: is_moa=True, is_gqa=True, 32 query heads / 16 K/V heads x 128 dim (2 query heads per K/V head), 16 query heads per attention

In [8]:
all_traces = []
for domain, prompts in PROMPTS.items():
    for prompt in prompts:
        print(f"[{len(all_traces) + 1}/{total_prompts}] ({domain}) extracting: {prompt!r}")
        trace = extract_for_prompt(prompt)
        trace["domain"] = domain
        all_traces.append(trace)
        print(f"    tokens={[t['text'] for t in trace['tokens']]}  top_pred={trace['next_token_candidates'][0]}")

with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump({"prompts": all_traces}, f)

print(f"\nWrote {len(all_traces)} prompts to {OUT_PATH} ({os.path.getsize(OUT_PATH) / 1e6:.2f} MB)")

[1/12] (code) extracting: 'The most popular programming language for data science is'
    tokens=['<s>', 'The', 'most', 'popular', 'programming', 'language', 'for', 'data', 'science', 'is']  top_pred={'token': 'Python', 'prob': 0.745738}
[2/12] (code) extracting: 'A loop that never terminates is called an infinite'
    tokens=['<s>', 'A', 'loop', 'that', 'never', 'term', 'inates', 'is', 'called', 'an', 'infinite']  top_pred={'token': 'loop', 'prob': 0.984112}
[3/12] (math) extracting: 'The square root of sixteen is'
    tokens=['<s>', 'The', 'square', 'root', 'of', 'sixteen', 'is']  top_pred={'token': 'four', 'prob': 0.898073}
[4/12] (math) extracting: 'Two plus two equals'
    tokens=['<s>', 'Two', 'plus', 'two', 'equals']  top_pred={'token': 'five', 'prob': 0.434956}
[5/12] (biomedical) extracting: 'The organ that pumps blood throughout the human body is the'
    tokens=['<s>', 'The', 'organ', 'that', 'p', 'umps', 'blood', 'throughout', 'the', 'human', 'body', 'is', 'the']  top_pred=

## UMAP: per-domain FFN expert activation

Reuses the top-k routing decisions already captured in `all_traces` above (no extra forward
passes), same method as `extract_routing_trace.ipynb` cell 8. FFN-side only for now --
attention-MoA gets its own 8x24 activation grid and could get its own UMAP the same way, but
is left as a future extension since there's no second-router UI to feed it yet either.

In [9]:
import umap

NUM_LAYERS = num_layers
NUM_EXPERTS = ffn_num_experts
domain_to_idx = {d: i for i, d in enumerate(domains)}

# activation_counts[layer][expert][domain_idx] = count of this domain's tokens with this
# expert in top-k at this layer. tokens_per_domain[domain_idx] = total tokens seen for that
# domain (same at every layer, since every layer sees the same tokenized prompts).
activation_counts = np.zeros((NUM_LAYERS, NUM_EXPERTS, len(domains)), dtype=np.float64)
tokens_per_domain = np.zeros(len(domains), dtype=np.float64)

expert_token_scores = defaultdict(list)  # (layer, expert) -> [(weight, token, domain, prompt)]

for trace in all_traces:
    d_idx = domain_to_idx[trace["domain"]]
    n_tokens = len(trace["tokens"])
    tokens_per_domain[d_idx] += n_tokens
    for layer_trace in trace["layers"]:
        li = layer_trace["layer"]
        for tt in layer_trace["tokens"]:
            token_text = trace["tokens"][tt["token_index"]]["text"]
            for e, w in zip(tt["top_experts"], tt["top_weights"]):
                activation_counts[li, e, d_idx] += 1
                expert_token_scores[(li, e)].append((w, token_text, trace["domain"], trace["prompt"]))

tokens_per_domain[tokens_per_domain == 0] = 1  # guard divide-by-zero
activation_rates = activation_counts / tokens_per_domain[None, None, :]
expert_vectors = activation_rates.reshape(NUM_LAYERS * NUM_EXPERTS, len(domains))

point_layer_ids = np.repeat(np.arange(NUM_LAYERS), NUM_EXPERTS)
point_expert_ids = np.tile(np.arange(NUM_EXPERTS), NUM_LAYERS)

# Never-activated (layer, expert) pairs are all-zero and undefined under the cosine metric
# (0/0 -> NaN) -- exclude from the projection, report separately as excluded_experts.
active_mask = expert_vectors.sum(axis=1) > 0
active_vectors = expert_vectors[active_mask]

reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1, metric="cosine", n_jobs=1)
active_embedding = reducer.fit_transform(active_vectors)

assert not np.isnan(active_embedding).any(), (
    "UMAP produced NaN coordinates even after excluding all-zero rows -- inspect "
    "active_vectors for degenerate rows, or re-run with metric='euclidean'."
)

print(f"Built {expert_vectors.shape[0]} (layer, expert) vectors across {len(domains)} domains.")
print(f"UMAP embedding shape: {active_embedding.shape} ({int(active_mask.sum())} active of {expert_vectors.shape[0]} total pairs)")

TOP_K_TOKENS = 8
active_indices = np.flatnonzero(active_mask)

umap_points = []
for row, i in enumerate(active_indices):
    layer_id = int(point_layer_ids[i])
    expert_id = int(point_expert_ids[i])
    vec = expert_vectors[i]
    dominant_domain = domains[int(np.argmax(vec))]

    samples = sorted(expert_token_scores.get((layer_id, expert_id), []), key=lambda item: -item[0])[:TOP_K_TOKENS]
    top_tokens = [
        {"token": tok, "score": round(float(score), 4), "domain": dom, "prompt": prompt}
        for score, tok, dom, prompt in samples
    ]

    umap_points.append({
        "layer_id": layer_id,
        "expert_id": expert_id,
        "x": round(float(active_embedding[row, 0]), 4),
        "y": round(float(active_embedding[row, 1]), 4),
        "dominant_domain": dominant_domain,
        "domain_activation_rate": {d: round(float(vec[j]), 4) for j, d in enumerate(domains)},
        "top_tokens": top_tokens,
    })

excluded_experts = [
    {"layer_id": int(point_layer_ids[i]), "expert_id": int(point_expert_ids[i])}
    for i in np.flatnonzero(~active_mask)
]

assert len(umap_points) + len(excluded_experts) == NUM_LAYERS * NUM_EXPERTS

umap_data = {
    "domains": domains,
    "num_layers": NUM_LAYERS,
    "num_experts": NUM_EXPERTS,
    "points": umap_points,
    "excluded_experts": excluded_experts,
}

with open(UMAP_OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(umap_data, f, ensure_ascii=False, allow_nan=False, indent=2)

print(f"Wrote {UMAP_OUT_PATH} ({len(umap_points)} points, {len(excluded_experts)} excluded pairs)")

Built 192 (layer, expert) vectors across 6 domains.
UMAP embedding shape: (192, 2) (192 active of 192 total pairs)
Wrote jetmoe_routing_trace_umap.json (192 points, 0 excluded pairs)


In [10]:
try:
    from google.colab import files
    files.download(OUT_PATH)
    files.download(UMAP_OUT_PATH)
except ImportError:
    print("Not running in Google Colab -- skipping auto-download.")
    print(f"Files were written locally at: {OUT_PATH} and {UMAP_OUT_PATH}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>